# 02 — Custom CNN Training (from scratch)

Trains the 4-block custom CNN defined in `src/models.py` on the identity subset
prepared in `01_data_preparation.ipynb`. Saves the best checkpoint, training curves,
and history to `logs/`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
from torch.utils.data import DataLoader

from dataset import CelebASubset
from models import CustomCNN
from utils import set_seed, get_device, train_model

set_seed(42)
device = get_device()
print("device:", device)


device: cpu


In [2]:
BATCH_SIZE = 16
EPOCHS = 40
LR = 3e-4

train_ds = CelebASubset("train")
val_ds = CelebASubset("val")
num_classes = train_ds.num_classes
print("num_classes:", num_classes, "| train:", len(train_ds), "| val:", len(val_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


num_classes: 4 | train: 67 | val: 16


In [3]:
model = CustomCNN(num_classes=num_classes)
print(f"CustomCNN parameters: {model.num_params():,}")


CustomCNN parameters: 1,207,588


In [4]:
history, train_time_sec, best_val_acc = train_model(
    model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR, device=device, run_name="custom_cnn", patience=10,
)
print(f"\nBest val accuracy: {best_val_acc:.4f}  |  Training time: {train_time_sec:.1f}s")


[custom_cnn] epoch  1/40  train_loss=1.3858 train_acc=0.269  val_loss=1.3873 val_acc=0.250
[custom_cnn] epoch  2/40  train_loss=1.2750 train_acc=0.433  val_loss=1.3821 val_acc=0.250
[custom_cnn] epoch  3/40  train_loss=1.1800 train_acc=0.478  val_loss=1.3734 val_acc=0.438
[custom_cnn] epoch  4/40  train_loss=1.1466 train_acc=0.582  val_loss=1.3317 val_acc=0.438
[custom_cnn] epoch  5/40  train_loss=1.1348 train_acc=0.537  val_loss=1.2780 val_acc=0.312
[custom_cnn] epoch  6/40  train_loss=1.0360 train_acc=0.582  val_loss=1.2273 val_acc=0.375
[custom_cnn] epoch  7/40  train_loss=0.9800 train_acc=0.642  val_loss=1.2024 val_acc=0.438
[custom_cnn] epoch  8/40  train_loss=0.9496 train_acc=0.642  val_loss=1.3351 val_acc=0.375
[custom_cnn] epoch  9/40  train_loss=0.9295 train_acc=0.701  val_loss=1.2023 val_acc=0.375
[custom_cnn] epoch 10/40  train_loss=0.8866 train_acc=0.746  val_loss=0.9767 val_acc=0.562
[custom_cnn] epoch 11/40  train_loss=0.8273 train_acc=0.731  val_loss=0.9654 val_acc=0.625

**Notes for CPU-only runs (Explorer):** if a compute node kernel dies with no
traceback while this cell runs, that node likely lacks AVX2 — request a different
node/partition, or drop `BATCH_SIZE`/`IMAGE_SIZE` further. 25 epochs on ~5 classes at
128x128 should complete in well under an hour on a modern CPU; reduce `EPOCHS` if
iterating.